In [ ]:
import holoviews as hv
import xarray as xr
import hvplot.xarray
import matplotlib.pylab as plt
from matplotlib.colors import Normalize

import numpy as np
import panel as pn

import cmcrameri.cm  # noqa: F401  pylint: disable=unused-import
import cmglaciology.cm  # noqa: F401  pylint: disable=unused-import
import cmcrameri.cm as cmc

from pism_terra.plotting import blended_animation


In [ ]:
ds = []
for m in ["mean", "max"]:
    _ds = xr.open_dataset(f"/Users/andy/base/pism-terra/2026_10_plume/output/processed_scalar/{m}_GIS_JIB_g900m_id_OCX_ocx_free_1985-01-01_2005-01-01.nc")
    _ds = _ds.expand_dims({"exp_id": [m]})
    ds.append(_ds)
ds = xr.concat(ds, dim="exp_id").set_index({"glacier_id":"glacier_id_name"}).sel({"glacier_id": "JIB_SHELF"})
ds.glacier_id
ds["glf"] = -ds["ice_mass_transport_across_grounding_line"]

In [ ]:
spatial = xr.open_dataset("/Users/andy/base/pism-terra/2026_10_plume/output/processed_spatial/spatial_GIS_JIB_g900m_id_OCX_ocx_free_1985-01-01_2005-01-01.nc")
spatial = spatial.sel({"time": slice("1992", "2005")})

In [ ]:
blended_animation(spatial.velsurf_mag, spatial.usurf, cmap="cmg.speed", clim=(0, 10000))

In [ ]:
blended_animation(spatial.plume_basal_melt_rate, spatial.usurf, cmap="cmc.davos_r", clim=(0, 500))


In [ ]:
blended_animation(spatial.thk, spatial.usurf, cmap="cmc.lipari_r", clim=(0, 500))

In [ ]:
std = (ds.drop_vars(["pism_config"]) - ds.drop_vars(["pism_config"]).min()) / (ds.drop_vars(["pism_config"]).max() - ds.drop_vars(["pism_config"]).min())
mean_scaled = (std * (1 - -1) + -1).sel(exp_id="mean").sel(time=slice("1992", "2002")).rolling(time=13).mean()

fields = ds.drop_vars(["pism_config"], errors="ignore")
fields = fields[[v for v in fields.data_vars if fields[v].dtype.kind == "f"]]  # floats only

lo, hi = fields.min(), fields.max()          # add a dim, e.g. .min("time"), to scale per cell
span = (hi - lo).where(hi > lo)              # constant variables become NaN instead of inf
scaled = 2 * (fields - lo) / span - 1
mean_scaled = scaled.sel(exp_id="mean").sel(time=slice("1992", "2002"))

In [ ]:
import matplotlib.pylab as plt
import numpy as np

In [ ]:
fig, ax = plt.subplots(1, 1, sharex=True, figsize=(12, 8))
vs = ["eigen1", "glf", "plume_basal_melt_rate", "thk", "frontal_melt_retreat_rate"]
ls = ["eigen", "D", "m", "H", "fm"]
cs = cmc.managua(np.linspace(0.1, 0.8, len(vs)))
for _v, _label, _color in zip(vs, ls, cs):
    mean_scaled[_v].plot(ax=ax, lw=0.75, color=_color, ls="dashed")
    mean_scaled[_v].rolling(time=7, center=True).mean().plot(ax=ax, lw=2.5, label=_label, color=_color)
ax.axhline(ls="dotted", lw=0.5)
ax.legend()
ax.set_title("Normalized")
ax.set_ylabel(None)

In [ ]:
ds.frontal_melt_rate.values

In [ ]:
ds.data_vars

In [ ]:
import pandas as pd
df = pd.read_csv("/Users/andy/Downloads/jib_fractures.csv")

In [ ]:
df.max() - df.min()

In [ ]:
df.max()

In [ ]:
df.min()

In [ ]:
1-4.5e-2

In [ ]:
1-2e-2

In [ ]:
1.4e5-4e4

In [ ]:
import geopandas as gpd

In [ ]:
df = gpd.read_file("/Users/andy/base/pism-terra/2026_10_qaamerujup_prepare/input/mouginot_glaciers_w_shelves_s.gpkg")

In [ ]:
df["glacier_id"] = "GIS_" + df["NAME"]

In [ ]:
df.to_file("/Users/andy/base/pism-terra/2026_10_qaamerujup_prepare/input/mouginot_glaciers_w_shelves_s.gpkg")

In [ ]:
df

In [ ]:
gpd.read_file?